# Расчёт динамики, структурных долей и макронагрузки налоговой задолженности РФ

В этом блокноте выполняются основные аналитические расчёты по очищенному датасету `passport_debt_rf_2020_2022.csv`. Рассчитываются темпы прироста за 2020–2022 годы, удельный вес отдельных налогов в совокупном долге, распределение по статусам взыскания и отношение долга к доходам консолидированного бюджета РФ.


## Шаг 1. Расчёт темпов прироста, структуры долга и бюджетной нагрузки
Формируем три сводные аналитические таблицы:
1. Динамика абсолютных значений (в млрд руб.) и относительных приростов за 2020–2021, 2021–2022 и 2020–2022 годы.
2. Структурные доли видов налогов, пеней и страховых взносов в совокупной задолженности РФ (строка 823).
3. Доли статусов взыскания и отношение совокупного долга к доходам консолидированного бюджета РФ.


In [1]:
import os
import pandas as pd

df = pd.read_csv(os.path.join(r"..\data_processed", "passport_debt_rf_2020_2022.csv"))
df["Код_строки"] = df["Код_строки"].astype(str).str.strip()
df["Всего_млрд_руб"] = df["Всего_млн_руб"] / 1000

# 1. ДИНАМИКА И ТЕМПЫ ПРИРОСТА ПО ВСЕМ ПОКАЗАТЕЛЯМ РФ (МЛРД РУБ. И %)
pivot_dyn = df.pivot(index=["Код_строки", "Показатель"], columns="Год", values="Всего_млрд_руб").reset_index()

c20 = pivot_dyn.get(2020)
c21 = pivot_dyn.get(2021)
c22 = pivot_dyn.get(2022)

pivot_dyn["Прирост_20_21_%"] = ((c21 - c20) / c20 * 100).round(1)
pivot_dyn["Прирост_21_22_%"] = ((c22 - c21) / c21 * 100).round(1)
pivot_dyn["Прирост_20_22_млрд"] = (c22 - c20).round(2)
pivot_dyn["Прирост_20_22_%"] = ((c22 - c20) / c20 * 100).round(1)

for y in (2020, 2021, 2022):
    pivot_dyn[y] = pivot_dyn.get(y).round(2)

print("=== 1. ДИНАМИКА ЗАДОЛЖЕННОСТИ ПО РФ (МЛРД РУБ. И %) ===")
print(pivot_dyn.to_string(index=False))

# 2. СТРУКТУРНЫЕ ДОЛИ КАТЕГОРИЙ В СОВОКУПНОМ ДОЛГЕ РФ (ОТ СТРОКИ 823)
df_823 = df[df["Код_строки"] == "823"].set_index("Год")
totals_map = df_823["Всего_млн_руб"].to_dict()

df["Доля_от_823_%"] = df.apply(lambda r: round(r["Всего_млн_руб"] / totals_map[r["Год"]] * 100, 1), axis=1)
pivot_shares = df.pivot(index=["Код_строки", "Показатель"], columns="Год", values="Доля_от_823_%").reset_index()

print("\n=== 2. ДОЛЯ КАЖДОЙ КАТЕГОРИИ В СОВОКУПНОМ ДОЛГЕ РФ (%) ===")
print(pivot_shares.to_string(index=False))

# 3. СТАТУСЫ ВЗЫСКАНИЯ И МАКРО-НАГРУЗКА К ДОХОДАМ БЮДЖЕТА РФ (%)
macro = pd.DataFrame(index=df_823.index)
macro["Всего_трлн_руб"] = (df_823["Всего_млн_руб"] / 1_000_000).round(2)
macro["Недоимка_и_пени_%"] = (df_823["Недоимка_и_пени_млн_руб"] / df_823["Всего_млн_руб"] * 100).round(1)
macro["Приостановленные_%"] = (df_823["Приостановленные_млн_руб"] / df_823["Всего_млн_руб"] * 100).round(1)
macro["Отсроченные_%"] = (df_823["Отсроченные_млн_руб"] / df_823["Всего_млн_руб"] * 100).round(1)
macro["Долг_к_доходам_КБ_РФ_%"] = (df_823["Всего_млн_руб"] / df_823["Поступило_доходов_КБ_РФ_млн_руб"] * 100).round(2)

print("\n=== 3. СТРУКТУРА ПО СТАТУСАМ И МАКРО-НАГРУЗКА ПО РФ ===")
print(macro.reset_index().to_string(index=False))


=== 1. ДИНАМИКА ЗАДОЛЖЕННОСТИ ПО РФ (МЛРД РУБ. И %) ===
Код_строки                                                  Показатель    2020    2021    2022  Прирост_20_21_%  Прирост_21_22_%  Прирост_20_22_млрд  Прирост_20_22_%
       801                   Задолженность по налогам и сборам (всего) 1079.76 1195.34 1617.59             10.7             35.3              537.83             49.8
       802                                  Федеральные налоги и сборы  794.03  914.99 1332.26             15.2             45.6              538.23             67.8
       804                                Налог на прибыль организаций  149.67  217.71  320.37             45.5             47.2              170.70            114.0
       806                          НДС (по товарам, реализуемым в РФ)  483.05  526.14  725.43              8.9             37.9              242.37             50.2
       807                                                        НДПИ   24.26   35.03  114.60             44.4   